# Post-processing

## Strut'n'Tie — LUSAS LPI Course Series  
### Video 05

**Author:** Kamil Riedel  
**© Strut'n'Tie**  
**License:** MIT Licence

# Connect to LUSAS

In [2]:
from shared.LPI import *
import shared.Helpers as Helpers
import pandas as pd
import numpy as np

modeller = get_lusas_modeller()
Helpers.initialise(modeller)

if not modeller.existsDatabase():
    raise Exception("A model must be open before running this code")

database = modeller.database()

# Define variables

In [3]:
LOADCASE_SW = "Self-weight"
LOADCASE_DL = "Dead load"
LOADCASE_LL = "Live load"

LINE_ID = "Line ID"
SECTION_NAME = "Section name"
AREA = "Area"
I_YY = "Iyy"
I_ZZ = "Izz"
LENGTH = "Length"
ELEMENT_ID = "Element ID"
F_ED = "F_Ed"
F_RD = "F_Rd"
F_RD_TENSION = "F_Rd_ten"
F_RD_COMPRESSION = "F_Rd_comp"
UTILISATION = "Utilisation"

# Helper fucntions

In [4]:
# Euclidian line length
def line_length(startX,startY,startZ,endX,endY,endZ):  
    dx = endX - startX
    dy = endY - startY
    dz = endZ - startZ
    return np.sqrt(dx*dx + dy*dy + dz*dz)


# Extract data

## Geometric sections

In [5]:
# ================================================================== #
# GEOMETRIC ATTRIBUTES FROM DATABASE #
data = []
attributes = database.getAttributes("Geometric")

# attr = attributes[0]
# print(f"attribute name: {attr.getName()}\n")
# print("Value names:")
# for value_name in attr.getValueNames():
#     print(value_name)

for attr in attributes:
    data.append({SECTION_NAME: attr.getName(), AREA: attr.getValue("A"), I_YY: attr.getValue("Iyy"), I_ZZ: attr.getValue("Izz")})

sections_df = pd.DataFrame(data)
sections_df

,Section name,Area,Iyy,Izz
0,UC254,0.011331,0.000143,0.000049
1,UC203,0.007637,0.000061,0.000021
2,203UC60,0.015096,0.000121,0.000165
3,254UC89,0.022385,0.000282,0.000389
4,RL,1.000000,0.083333,0.083333


In [6]:
# ================================================================== #
# GEOMETRIC ATTRIBUTES FROM FEATURES #

line = database.getObject("Lines", 1) # Get line 1
assignments = line.getAssignments("Geometric") # Get geometric assignments
attr = assignments[0].getAttribute() # Get geometric attribute
print(f"attribute name: {attr.getName()}\n")
print("Value names:")
for value_name in attr.getValueNames():
    print(value_name)

attribute name: 254UC89

Value names:
elementType
isNodal
reinforcement
A
Iyy
Izz
Iyz
J
I'yy
I'zz
I'yz
J'
ez0
ey0
ez
ey
Iyr
Izr
Irr
Iwr
external perimeter
internal perimeter
Material cy
Material cz
Asz
Asy
Epw
Ap
Zpy
Zpz
yp
zp
Zpt
Cw
yo
zo
betay
betaz
Imax
Imin
ky
kz
kmin
Syt
Syb
Szt
Szb
isTapered
Rotation
Mirrored
yt
yb
zt
zb
FibreNames
FibreCoords
FibreFillTypes
Type
B
D
tf
tw
r


## Lines

In [7]:
# ================================================================== #
# LINE - SECTIONS - LENGTH #
lines = database.getObjects("Lines")

data = []

for line in lines:
    lineID = line.getName()
    section = line.getAssignments("Geometric")
    section = section[0].getAttribute()
    sectName = section.getName()
    # Line length
    startNode = line.getStartPoint()
    endNode = line.getEndPoint()
    startX,startY,startZ = startNode.getXYZ()
    endX,endY,endZ = endNode.getXYZ()
    length = line_length(startX,startY,startZ,endX,endY,endZ)
    data.append({LINE_ID: lineID, SECTION_NAME: sectName, LENGTH: length})

lines_df = pd.DataFrame(data)
lines_df


,Line ID,Section name,Length
0,1,254UC89,6.5


## Elements

In [ ]:
# ================================================================== #
# GET ANALYSES #

# Results context 
context = modeller.newResultsContext(None)
# Add all elements
elements : list[IFElement] = database.getObjects("Element")
context.getCalcResultsSet().add(elements)
# Get loadsets
loadcase_SW = database.getLoadset(LOADCASE_SW)
loadcase_DL = database.getLoadset(LOADCASE_DL)
loadcase_LL = database.getLoadset(LOADCASE_LL)
# Ways to set loadset
context.setActiveLoadset(1) # Set using a loadset ID
context.setActiveLoadset(loadcase_SW) # Set using a loadset reference

com_error: (-2147352567, 'Exception occurred.', (0, 'Application Database', '"Loadcase 1" is not the name of an existing Loadset', None, 0, -2147467259), None)

In [ ]:
# ================================================================== #
# DF: ELEMENTS - LINES - FORCES #

data = []

# # Get individual element results - inefficient
element = elements[0]
context.setActiveLoadset(loadcase_DL)
fx = element.getInternalResults(0, "Force/Moment - Thick 3D Beam", "Fx", context)
print(fx)

# Get the internal point results for all beam elements
context.setActiveLoadset(loadcase_SW)
results_SW = database.getResultsComponentSet("Force/Moment - Thick 3D Beam", "Fx", "Internal", context)
context.setActiveLoadset(loadcase_DL)
results_DL = database.getResultsComponentSet("Force/Moment - Thick 3D Beam", "Fx", "Internal", context)
context.setActiveLoadset(loadcase_LL)
results_LL = database.getResultsComponentSet("Force/Moment - Thick 3D Beam", "Fx", "Internal", context)
i_fx = results_SW.getComponentNumber("Fx") # Component number for fx dof

# fx = results_SW.getInternalResults(i_fx, element, 0, None, None)
# print(fx)
# fx = results_DL.getInternalResults(i_fx, element, 0, None, None)
# print(fx)
# fx = results_LL.getInternalResults(i_fx, element, 0, None, None)
# print(fx)


for element in elements:
    fx_SW = results_SW.getInternalResults(i_fx, element, 0, None, None)
    fx_DL = results_DL.getInternalResults(i_fx, element, 0, None, None)
    fx_LL = results_LL.getInternalResults(i_fx, element, 0, None, None)
    fx_ULS = 1.35 * (fx_SW + fx_DL) + 1.5 * fx_LL
    data.append({ELEMENT_ID: element.getName(), LINE_ID: element.getFeature().getName(), F_ED: fx_ULS})


element_df = pd.DataFrame(data)
element_df

NameError: name 'elements' is not defined

# Process results

Critical strength due to buckling:
$P_{cr} = (\pi^2  E  I) / (L^2)$

Critical strength due compressive resistance:
$P_{cr} = fy A$

In [ ]:
element_df

,Element ID,Line ID,F_Ed
0,1,1,-16800.000000
1,2,1,-16800.000000
2,3,1,-16800.000000
3,4,2,-4200.000000
4,5,2,-4200.000000
...,...,...,...
100,101,34,16099.689438
101,102,34,16099.689438
102,103,35,-16099.689438
103,104,35,-16099.689438


In [ ]:
lines_df


,Line ID,Section name,Length
0,1,bottom chord,3.000000
1,2,bottom chord,3.000000
2,3,bottom chord,3.000000
3,4,bottom chord,3.000000
4,5,bottom chord,3.000000
5,6,bottom chord,3.000000
6,7,bottom chord,3.000000
7,8,bottom chord,3.000000
8,9,bottom chord,3.000000
9,10,top chord,3.000000


In [ ]:
sections_df

,Section name,Area,Iyy,Izz
0,bottom chord,0.009310,0.000114,0.000039
1,top chord,0.009310,0.000114,0.000039
2,diagonals,0.003826,0.000017,0.000006


In [ ]:
# Add line resistance

fy = 355E6 # Pa
E = 205E9 # Pa

# Merge relevant section properties into line_df based on Section name
line_resistances = lines_df.merge(sections_df, on="Section name", how="left")

# Calculate resistance
resistances_tension = [fy * row[AREA] for index, row in line_resistances.iterrows()]

resistances_compression = [min(
        fy * row[AREA],
        np.pi**2 * E * row[I_YY] / row[LENGTH]**2,
        np.pi**2 * E * row[I_ZZ] / row[LENGTH]**2
    ) for index, row in line_resistances.iterrows()]

line_resistances[F_RD_TENSION] = resistances_tension
line_resistances[F_RD_COMPRESSION] = resistances_compression

line_resistances

,Line ID,Section name,Length,Area,Iyy,Izz,F_Rd_ten,F_Rd_comp
0,1,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
1,2,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
2,3,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
3,4,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
4,5,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
5,6,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
6,7,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
7,8,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
8,9,bottom chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06
9,10,top chord,3.000000,0.009310,0.000114,0.000039,3.305090e+06,3.305090e+06


In [ ]:
# Retrieve tensile resistance for element 3
line_resistances.loc[line_resistances[LINE_ID] == "3", F_RD_TENSION].iloc[0]

np.float64(3305089.9449203773)

In [ ]:
# Add element resistance

# Merge relevant section properties into line_df based on Section name
element_resistances = element_df.merge(line_resistances[[LINE_ID, F_RD_TENSION, F_RD_COMPRESSION]], on=LINE_ID, how="left")

# Add utilisation
utilisation = [
    np.abs(row[F_ED] / row[F_RD_TENSION]) 
    if row[F_ED] > 0 
    else np.abs(row[F_ED] / row[F_RD_COMPRESSION])
    for index, row in element_resistances.iterrows()
]

element_resistances[UTILISATION] = utilisation

element_resistances

,Element ID,Line ID,F_Ed,F_Rd_ten,F_Rd_comp,Utilisation
0,1,1,-16800.000000,3.305090e+06,3.305090e+06,0.005083
1,2,1,-16800.000000,3.305090e+06,3.305090e+06,0.005083
2,3,1,-16800.000000,3.305090e+06,3.305090e+06,0.005083
3,4,2,-4200.000000,3.305090e+06,3.305090e+06,0.001271
4,5,2,-4200.000000,3.305090e+06,3.305090e+06,0.001271
...,...,...,...,...,...,...
100,101,34,16099.689438,1.358337e+06,1.007972e+06,0.011852
101,102,34,16099.689438,1.358337e+06,1.007972e+06,0.011852
102,103,35,-16099.689438,1.358337e+06,1.007972e+06,0.015972
103,104,35,-16099.689438,1.358337e+06,1.007972e+06,0.015972


In [ ]:
# Get maximum utilisation
print(f"max utilisation = {max(element_resistances[UTILISATION])*100:.2f}%")
element_resistances.loc[ element_resistances[UTILISATION] == max(element_resistances[UTILISATION]) ]

max utilisation = 1.60%


,Element ID,Line ID,F_Ed,F_Rd_ten,F_Rd_comp,Utilisation
104,105,35,-16099.689438,1.358337e+06,1.007972e+06,0.015972


In [ ]:
# Get tonnage
density = 7850 # kg/m^3
mass = (line_resistances[LENGTH] * line_resistances[AREA]).sum() * density
print(f"tonnage = {mass:.2e} kg")


tonnage = 5.54e+03 kg
